In [1]:
CONFIG_NAME = "wav2vec2.yaml"
AUDIO_PATH = "acoustic/data/sound.wav"

In [2]:
import sys
from pathlib import Path
import torch
import librosa
import logging

sys.path.insert(0, str(Path.cwd()))

from acoustic.utils.config import load_config
from acoustic.models.load_model import build_model
from acoustic.models import get_generate_method

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
config_path = f"acoustic/configs/{CONFIG_NAME}"
cfg = load_config(config_path)

In [4]:
output_dir = Path(cfg['training']['output_dir'])
final_dir = output_dir / "final_model"

if final_dir.exists():
    checkpoint_dir = final_dir
    logger.info("Using final model")
else:
    checkpoints = sorted(output_dir.glob("checkpoint-*"))
    if not checkpoints:
        raise FileNotFoundError(
            f"No checkpoints or final model found in {output_dir}. Train the model first."
        )
    checkpoint_dir = max(checkpoints, key=lambda p: int(p.name.split("-")[-1]))
    logger.info(f"Using latest checkpoint: {checkpoint_dir}")

model, processor, _ = build_model(cfg)
model = model.from_pretrained(str(checkpoint_dir))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

INFO:__main__:Using final model
INFO:acoustic.models.load_model:Builder 'wav2vec2' not registered, trying to import package acoustic.models.wav2vec2
/home/abonentvneseti/programming/github/STT_russian_lang/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-russian were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [5]:
default_audio = AUDIO_PATH
test_cfg = cfg.get('test', {})
audio_path = test_cfg.get('audio_path', None)

if audio_path is None:
    print(f"No 'test.audio_path' found in config. Using default: {default_audio}")
    audio_path = default_audio

try:
    audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)
except Exception as e:
    logger.error(f"Failed to load audio from '{audio_path}': {e}")
    raise

No 'test.audio_path' found in config. Using default: acoustic/data/sound.wav


In [6]:
inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
input_key = "input_features" if "input_features" in inputs else "input_values"
input_data = inputs[input_key].to(device)

builder_key = cfg['model']['builder']
generate_fn = get_generate_method(builder_key)

with torch.no_grad():
    predicted_ids = generate_fn(model, input_data, processor)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("\nРаспознанный текст:", transcription)


Распознанный текст: допустим оно должно работать
